In [ ]:
import os
import time
import pandas as pd
from io import StringIO
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select, WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import (
    NoSuchElementException,
    TimeoutException,
    StaleElementReferenceException,
    ElementClickInterceptedException,
)
from selenium.webdriver.chrome.service import Service as ChromeService
from webdriver_manager.chrome import ChromeDriverManager  # optional convenience

# -----------------------------
# CONFIG
# -----------------------------
URL = "https://traffic.dot.ga.gov/ATSPM/"
OUTPUT_FILE = "output/tmc_all_signals_3.csv" # change this based on group name
FAILED_DIR = "output/failed_pages"
os.makedirs("output", exist_ok=True)
os.makedirs(FAILED_DIR, exist_ok=True)


#GROUP 0
# SIGNAL_IDS = ["4284", "7769", "11481", "11190", "11316", "11461", "11150", "11334", "7547", "7604",
#        "4095041", "4095114", "6115052", "6115041", "8591", "161", "7714", "7113", "39", "178"]

# GROUP 1
# SIGNAL_IDS = [
#    "4095107", "1359", "2130", "6028", "5045", "5051307", "6079",
#    "5051458", "6057045", "5051126", "7632", "7580", "1434", "7864", "7067099"
# ]

# # GROUP 2
# SIGNAL_IDS = ["196", "7691", "8079", "1475", "7913", "7007", "8052", "7803", "1351385", "5127209",
#        "5127204", "1896", "1144", "11313"]

# GROUP 3
SIGNAL_IDS = ["7662", "2051", "3215274", "3215133", "3215126", "3215053", "6223049", "6668",
       "3246", "8043", "4200", "5285", "5011"]



START_DATE = "11/02/2025"
START_TIME = "12:00"
START_AMPM = "AM"
END_DATE = "11/08/2025"
END_TIME = "11:59"
END_AMPM = "PM"

# -----------------------------
# START BROWSER
# -----------------------------
options = webdriver.ChromeOptions()
# options.add_argument("--headless")  # uncomment if you want headless
options.add_argument("--start-maximized")
driver = webdriver.Chrome(service=ChromeService(ChromeDriverManager().install()), options=options)
wait = WebDriverWait(driver, 30)

driver.get(URL)
print("Loaded main page, giving it a few seconds to initialize...")
time.sleep(5)

all_dfs = []
failed_signals = []

# Helper: try clicking with multiple strategies
def safe_click(driver, by, selector, timeout=10):
    try:
        el = WebDriverWait(driver, timeout).until(EC.element_to_be_clickable((by, selector)))
        el.click()
        return True
    except Exception as e:
        # fallback: try click via JS if element found but not clickable
        try:
            el = driver.find_element(by, selector)
            driver.execute_script("arguments[0].click();", el)
            return True
        except Exception:
            return False

for sid in SIGNAL_IDS:
    print(f"\n--- Processing Signal {sid} ---")
    try:
        # 1) Ensure SignalID input exists, input value
        sig_input = wait.until(EC.presence_of_element_located((By.ID, "SignalID")))
        sig_input.clear()
        sig_input.send_keys(sid)
        time.sleep(0.5)

        # 2) Click the Select button - several fallback selectors
        clicked = False
        for selector_by, selector in [
            (By.ID, "selectButton"),
            (By.ID, "btnSignalID"),
            (By.CSS_SELECTOR, "input[type='button'][value='Select']"),
            (By.XPATH, "//input[@type='button' and (@value='Select' or contains(@class,'select'))]"),
            (By.XPATH, "//button[contains(., 'Select')]"),
        ]:
            try:
                el = driver.find_element(selector_by, selector)
                try:
                    el.click()
                except ElementClickInterceptedException:
                    driver.execute_script("arguments[0].click();", el)
                clicked = True
                print(f"Clicked select via {selector_by} '{selector}'")
                break
            except Exception:
                continue
        if not clicked:
            raise NoSuchElementException("Could not click the Select button (tried multiple selectors)")

        # small wait for chart section to appear
        time.sleep(1.0)

        # 3) Chart Selection: wait for MetricsList; choose Turning Movement Counts
        chart_elem = wait.until(EC.presence_of_element_located((By.ID, "MetricsList")))
        chart_select = Select(chart_elem)
        selected = False
        # Try by visible text first
        try:
            chart_select.select_by_visible_text("Turning Movement Counts")
            selected = True
            print("Selected 'Turning Movement Counts' by visible text")
        except Exception:
            # fallback: select by known value "5"
            try:
                chart_select.select_by_value("5")
                selected = True
                print("Selected 'Turning Movement Counts' by value '5'")
            except Exception:
                print("Could not select Turning Movement Counts (text or value). Will skip this signal.")
                raise NoSuchElementException("Turning Movement Counts option not available")

        time.sleep(1.0)

        # 4) Volume Bin Size -> select 60
        try:
            bin_elem = wait.until(EC.presence_of_element_located((By.ID, "SelectedBinSize")))
            Select(bin_elem).select_by_visible_text("60")
            print("Selected bin size 60")
        except Exception:
            print("Volume bin size element missing or cannot select 60")
            raise

        # 5) Ensure Show Data Table checked
        try:
            show_table = driver.find_element(By.ID, "ShowDataTable")
            if not show_table.is_selected():
                try:
                    show_table.click()
                except Exception:
                    driver.execute_script("arguments[0].click();", show_table)
            print("ShowDataTable ensured checked")
        except Exception:
            print("ShowDataTable element missing")
            raise

        # 6) Set start/end date/time values (use exact IDs provided)
        # Start
        driver.find_element(By.ID, "StartDateDay").clear()
        driver.find_element(By.ID, "StartDateDay").send_keys(START_DATE)
        driver.find_element(By.ID, "StartTime").clear()
        driver.find_element(By.ID, "StartTime").send_keys(START_TIME)
        Select(driver.find_element(By.ID, "StartAMPMddl")).select_by_visible_text(START_AMPM)
        # End
        driver.find_element(By.ID, "EndDateDay").clear()
        driver.find_element(By.ID, "EndDateDay").send_keys(END_DATE)
        driver.find_element(By.ID, "EndTime").clear()
        driver.find_element(By.ID, "EndTime").send_keys(END_TIME)
        Select(driver.find_element(By.ID, "EndAMPMddl")).select_by_visible_text(END_AMPM)
        print("Dates and times set")

        # 7) Click Create Chart (multiple fallbacks)
        created = False
        for selector_by, selector in [
            (By.ID, "CreateMetric"),
            (By.CSS_SELECTOR, "button#CreateMetric"),
            (By.XPATH, "//button[contains(., 'Create Chart')]"),
            (By.XPATH, "//button[contains(., 'Create Metric')]"),
        ]:
            try:
                el = driver.find_element(selector_by, selector)
                try:
                    el.click()
                except ElementClickInterceptedException:
                    driver.execute_script("arguments[0].click();", el)
                created = True
                print(f"Clicked Create Chart via {selector_by} '{selector}'")
                break
            except Exception:
                continue
        if not created:
            raise NoSuchElementException("Could not click Create Chart (tried multiple selectors)")

        # 8) Wait for the table to be present (longer wait)
        try:
            wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, "div.TMCTable table")))
        except TimeoutException:
            print("Timed out waiting for table to appear")
            raise

        # 9) Robustly fetch the table HTML with retries to avoid stale element
        html = None
        for attempt in range(5):
            try:
                table_div = driver.find_element(By.CSS_SELECTOR, "div.TMCTable table")
                html = table_div.get_attribute("outerHTML")
                if html and "<table" in html:
                    break
            except StaleElementReferenceException:
                print(f"Stale element, retrying ({attempt+1}/5)...")
                time.sleep(1)
            except Exception as e:
                print(f"Error getting table element: {e}")
                time.sleep(1)
        if not html:
            # Diagnostics: save page source
            failed_path = os.path.join(FAILED_DIR, f"failed_signal_{sid}.html")
            with open(failed_path, "w", encoding="utf-8") as f:
                f.write(driver.page_source)
            print(f"No table HTML could be extracted. Saved page for inspection: {failed_path}")
            failed_signals.append(sid)
            continue

        # Parse table via pandas (StringIO to avoid warning)
        df = pd.read_html(StringIO(html))[0]

        # -----------------------------
        # Flatten multi-row headers
        # -----------------------------
        if isinstance(df.columns, pd.MultiIndex):
            # join non-unnamed parts with underscore
            df.columns = [
                "_".join([str(c) for c in col if "Unnamed" not in str(c)])
                for col in df.columns.values
            ]
        # optional: replace spaces with underscores
        df.columns = [c.strip().replace(" ", "_") for c in df.columns]

        df["SignalID"] = sid
        all_dfs.append(df)
        print(f"✅ Collected data for {sid} (rows: {len(df)})")

        # polite pause
        time.sleep(1.0)

    except (NoSuchElementException, TimeoutException, StaleElementReferenceException) as e:
        print(f"⚠️ Skipped {sid}: {repr(e)}")
        # save page HTML for debugging
        failed_path = os.path.join(FAILED_DIR, f"failed_signal_{sid}.html")
        with open(failed_path, "w", encoding="utf-8") as f:
            f.write(driver.page_source)
        print(f"Saved page HTML to {failed_path}")
        failed_signals.append(sid)
        # continue with next signal
        continue
    except Exception as e:
        print(f"❌ Unexpected error for {sid}: {repr(e)}")
        failed_path = os.path.join(FAILED_DIR, f"failed_signal_{sid}_unexpected.html")
        with open(failed_path, "w", encoding="utf-8") as f:
            f.write(driver.page_source)
        failed_signals.append(sid)
        continue

# end loop
driver.quit()

# Combine results and write CSV
if all_dfs:
    tmc_all = pd.concat(all_dfs, ignore_index=True)
    tmc_all.to_csv(OUTPUT_FILE, index=False)
    print(f"\nSaved combined CSV to {OUTPUT_FILE} (total rows: {len(tmc_all)})")
else:
    print("\nNo data collected.")

if failed_signals:
    print("\nFailed signals (page-snapshots saved):")
    for fs in failed_signals:
        print(" -", fs)
else:
    print("\nNo failures.")


Loaded main page, giving it a few seconds to initialize...

--- Processing Signal 7662 ---
Clicked select via id 'selectButton'
Selected 'Turning Movement Counts' by visible text
Selected bin size 60
ShowDataTable ensured checked
Dates and times set
Clicked Create Chart via id 'CreateMetric'
✅ Collected data for 7662 (rows: 169)

--- Processing Signal 2051 ---
Clicked select via id 'selectButton'
Selected 'Turning Movement Counts' by visible text
Selected bin size 60
ShowDataTable ensured checked
Dates and times set
Clicked Create Chart via id 'CreateMetric'
✅ Collected data for 2051 (rows: 169)

--- Processing Signal 3215274 ---
Clicked select via id 'selectButton'
Could not select Turning Movement Counts (text or value). Will skip this signal.
⚠️ Skipped 3215274: NoSuchElementException()
Saved page HTML to output/failed_pages\failed_signal_3215274.html

--- Processing Signal 3215133 ---
Clicked select via id 'selectButton'
Could not select Turning Movement Counts (text or value). Wil